# Quantum Search in Graph Nodes - Tutorial 4: Understanding Quantum Circuits

This tutorial provides deep insights into how Grover's algorithm works at the quantum circuit level.

## Topics Covered
1. Quantum state initialization
2. Oracle construction
3. Diffusion operator
4. Circuit analysis and visualization
5. Amplitude amplification

## 1. Import and Setup

In [ ]:
from quantum.oracle import OracleBuilder
from quantum.diffusion import DiffusionOperator
from quantum.grover_search import GroverSearch
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
import numpy as np
import math

print("Quantum modules imported successfully!")
print(f"Using Qiskit Aer Simulator")

## 2. Understanding State Initialization

In [ ]:
# For 4 nodes, we need 2 qubits
n_qubits = 2
n_nodes = 2 ** n_qubits  # 4 nodes

print(f"System Parameters:")
print(f"  Nodes: {n_nodes}")
print(f"  Qubits required: {n_qubits}")
print(f"\nQuantum State Space:")
print(f"  Size: 2^{n_qubits} = {2**n_qubits} basis states")
print(f"  Basis states: |00⟩, |01⟩, |10⟩, |11⟩")
print(f"  Maps to nodes: 0, 1, 2, 3")

# Create superposition state
qc = QuantumCircuit(n_qubits, name='initialization')

# Hadamard gates create equal superposition
for i in range(n_qubits):
    qc.h(i)

print(f"\nInitial State (after Hadamards):")
print(f"  |ψ⟩ = (1/√{n_nodes}) × (|0⟩ + |1⟩ + |2⟩ + |3⟩)")
print(f"  Each basis state has amplitude: 1/√{n_nodes} ≈ {1/math.sqrt(n_nodes):.4f}")
print(f"  Probability of each: {1/n_nodes * 100:.1f}%")

print(f"\nCircuit:")
print(qc.draw())

## 3. Oracle Construction

In [ ]:
target = 2  # Binary: 10

print(f"Target Node: {target}")
print(f"Binary representation: {bin(target)[2:].zfill(n_qubits)}")
print(f"\nOracle Function:")
print(f"  Oracle(x) = -1 if x == {target}")
print(f"  Oracle(x) = +1 otherwise")
print(f"\nEffect on Quantum State:")
print(f"  Marks the target state with a phase of -1 (π phase flip)")
print(f"  This allows the amplitude to be amplified in subsequent steps")

# Build oracle
oracle_builder = OracleBuilder()
oracle = oracle_builder.construct_oracle(n_qubits, target)

print(f"\nOracle Circuit ({oracle.num_qubits} qubits):")
print(oracle.draw())
print(f"Gate count: {oracle.size()} gates")

## 4. Diffusion Operator

In [ ]:
print("Diffusion Operator (Amplitude Amplification):")
print(f"\nFormula: D = 2|s⟩⟨s| - I")
print(f"  where |s⟩ is the equal superposition state")
print(f"\nEffect:")
print(f"  1. Inverts amplitude around mean")
print(f"  2. Amplifies marked state amplitude")
print(f"  3. Suppresses unmarked states")
print(f"\nImplementation:")
print(f"  1. Apply Hadamards (basis change)")
print(f"  2. Apply controlled-Z on all qubits")
print(f"  3. Apply Hadamards again (basis change back)")

# Build diffusion operator
diffusion = DiffusionOperator()
diff_circuit = diffusion.construct_diffusion_operator(n_qubits)

print(f"\nDiffusion Circuit ({diff_circuit.num_qubits} qubits):")
print(diff_circuit.draw())
print(f"Gate count: {diff_circuit.size()} gates")

## 5. Complete Grover Iteration

In [ ]:
# Number of iterations needed
k = int(np.pi / 4 * np.sqrt(n_nodes))

print(f"Grover Iterations Required:")
print(f"  Formula: k ≈ π/4 × √N")
print(f"  Calculation: k ≈ π/4 × √{n_nodes} = {math.pi/4 * math.sqrt(n_nodes):.2f}")
print(f"  Rounded: k = {k}")

print(f"\nOne Grover Iteration Consists of:")
print(f"  1. Apply Oracle (marks target state)")
print(f"  2. Apply Diffusion Operator (amplifies amplitude)")

print(f"\nAmplitude Evolution:")
print(f"  Initial (uniform):")
amp_target_init = 1 / math.sqrt(n_nodes)
print(f"    Target:      {amp_target_init:.4f}")
print(f"    Others (×3): {amp_target_init:.4f}")

print(f"\n  After iteration 1:")
print(f"    Target amplitude increases")
print(f"    Other amplitudes decrease")

print(f"\n  After {k} iterations:")
print(f"    Target amplitude ≈ 1.0 (measurement probability ≈ 100%)")
print(f"    Other amplitudes ≈ 0.0 (negligible)")

## 6. Simulating One Grover Iteration

In [ ]:
# Manually construct one Grover iteration to visualize
qc_grover = QuantumCircuit(n_qubits, n_qubits)

# 1. Initialize superposition
for i in range(n_qubits):
    qc_grover.h(i)

# 2. One Grover iteration
# Apply oracle
oracle_circuit = oracle_builder.construct_oracle(n_qubits, target)
for instr in oracle_circuit.data:
    qc_grover.append(instr.operation, instr.qargs, instr.cargs)

# Apply diffusion
diff_circuit = diffusion.construct_diffusion_operator(n_qubits)
for instr in diff_circuit.data:
    qc_grover.append(instr.operation, instr.qargs, instr.cargs)

# 3. Measure
qc_grover.measure(range(n_qubits), range(n_qubits))

print("Complete Grover Circuit (1 iteration):")
print(qc_grover.draw())
print(f"\nTotal gates: {qc_grover.size()}")

## 7. Running Simulation

In [ ]:
# Simulate with ideal quantum computer
simulator = AerSimulator(method='statevector')
job = simulator.run(qc_grover.remove_final_measurements(copy=True), shots=1000)
result = job.result()
counts = result.get_counts()

print(f"Measurement Results (1000 shots):")
print(f"\nBasis State | Count | Probability")
print("-" * 40)

# Sort by count
sorted_counts = sorted(counts.items(), key=lambda x: x[1], reverse=True)

for state, count in sorted_counts:
    node = int(state, 2)  # Convert binary to decimal
    prob = count / 1000
    marker = "  ← TARGET" if node == target else ""
    print(f"|{state}⟩ ({node:1d}) | {count:4d}  | {prob*100:6.1f}%{marker}")

## 8. Full Grover Algorithm with All Iterations

In [ ]:
# Run complete Grover algorithm
grover = GroverSearch()
result = grover.search(n_nodes=n_nodes, target=target, shots=1000)

print(f"Complete Grover's Algorithm Results:")
print(f"-" * 50)
print(f"Target Node:           {target}")
print(f"Measured Node:         {result['measured_node']}")
print(f"Grover Iterations:     {result['grover_iterations']}")
print(f"Success Probability:   {result['success_probability']:.1f}%")
print(f"Execution Time:        {result['execution_time']*1000:.4f} ms")
print(f"\nResult: {'✓ SUCCESS' if result['found'] else '✗ FAILED'}")

## 9. Amplitude Amplification Visualization

In [ ]:
import matplotlib.pyplot as plt

# Theoretical amplitude after each iteration
iterations = list(range(k + 1))
target_amplitudes = []

# Using the theoretical formula for Grover amplitude
for i in iterations:
    # sin((2i+1)θ) where sin(θ) = 1/√N
    theta = np.arcsin(1 / np.sqrt(n_nodes))
    amp = np.sin((2*i + 1) * theta)
    target_amplitudes.append(amp)

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(iterations, target_amplitudes, 'b-o', linewidth=2, markersize=8, label='Target Amplitude')
ax.axhline(y=1.0, color='r', linestyle='--', alpha=0.5, label='Maximum')
ax.axhline(y=0.0, color='gray', linestyle='-', alpha=0.3)

ax.set_xlabel('Grover Iteration', fontsize=12)
ax.set_ylabel('Amplitude', fontsize=12)
ax.set_title(f'Amplitude Amplification: Target State vs Iterations (N={n_nodes})', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xticks(iterations)

plt.tight_layout()
plt.show()

print(f"Amplitude Evolution:")
for i, amp in enumerate(target_amplitudes):
    prob = amp ** 2
    print(f"  Iteration {i}: amplitude = {amp:6.4f}, probability = {prob*100:6.1f}%")

## 10. Key Quantum Computing Concepts

In [ ]:
print("Key Quantum Computing Concepts in Grover's Algorithm:")
print("=" * 60)

print("\n1. SUPERPOSITION")
print("-" * 40)
print(f"   Qubits can exist in linear combinations of states")
print(f"   Example: (1/√2)|0⟩ + (1/√2)|1⟩ (50% chance each)")
print(f"   With N qubits: search all {2**n_qubits} states simultaneously")

print("\n2. INTERFERENCE")
print("-" * 40)
print(f"   Amplitudes can cancel (destructive) or reinforce (constructive)")
print(f"   Oracle + Diffusion create interference patterns")
print(f"   Target amplitude amplified, others suppressed")

print("\n3. PHASE KICKBACK")
print("-" * 40)
print(f"   Oracle marks target with -1 phase (π phase flip)")
print(f"   Doesn't change measurement probability immediately")
print(f"   But enables amplification in next step")

print("\n4. AMPLITUDE AMPLIFICATION")
print("-" * 40)
print(f"   Diffusion operator reflects around mean amplitude")
print(f"   Target's amplitude increases exponentially")
print(f"   After k iterations, probability ≈ 100%")

print("\n5. MEASUREMENT")
print("-" * 40)
print(f"   Collapses superposition to one basis state")
print(f"   Probability = |amplitude|²")
print(f"   Multiple shots give statistical confidence")

print("\n" + "=" * 60)

## 11. Complexity Analysis

In [ ]:
print("Grover's Algorithm Complexity Analysis:")
print("=" * 60)

print("\nTIME COMPLEXITY:")
print("-" * 40)
print(f"  Query Complexity: O(√N)")
print(f"  Gate Complexity: O(√N × poly(log N))")
print(f"  Example: N={n_nodes} → √N ≈ 2 iterations")

print("\nSPACE COMPLEXITY:")
print("-" * 40)
print(f"  Qubits: O(log N)")
print(f"  Example: N={n_nodes} → log₂({n_nodes}) = {n_qubits} qubits")

print("\nCOMPARATIVE ADVANTAGE:")
print("-" * 40)
for n in [4, 16, 64, 256, 1024]:
    classical = n
    quantum = int(np.pi/4 * np.sqrt(n))
    speedup = classical / quantum
    qubits = int(np.log2(n))
    print(f"  N={n:4d}: Classical={classical:4d}, Quantum={quantum:2d}, Speedup={speedup:5.1f}x, Qubits={qubits}")

print("\n" + "=" * 60)

## 12. Summary

Grover's algorithm demonstrates quantum advantage through:

1. **Initialization**: Create equal superposition of all states
2. **Oracle**: Mark target state with phase flip (-1)
3. **Diffusion**: Amplify target amplitude, suppress others
4. **Iteration**: Repeat oracle + diffusion √N times
5. **Measurement**: Collapse to high-probability target state

**Result**: √N speedup - quadratic advantage over classical search!

## Further Reading
- Grover, L. K. (1996). "A fast quantum mechanical algorithm for database search"
- Qiskit Documentation: https://qiskit.org
- Nielsen & Chuang: "Quantum Computation and Quantum Information"